In [ ]:
# 1. Install the converter in this clean environment
!pip install tensorflowjs -q

# 2. Run the command-line converter
# !tensorflowjs_converter --input_format=keras /content/incep_model.keras /content/tfjs_model

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import preprocess_input
import shutil
import os

# Clean up our previous attempts
shutil.rmtree('/content/saved_model_dir', ignore_errors=True)
shutil.rmtree('/content/tfjs_model', ignore_errors=True)
!rm -f /content/tfjs_model.zip

print("1. Loading original model...")
model = tf.keras.models.load_model(
    '/content/incep_model.keras',
    custom_objects={'preprocess_input': preprocess_input}
)

print("2. Safely bypassing Data Augmentation layers...")
# We use clone_model to keep the complex Inception graph perfectly intact,
# simply swapping the augmentation layers for transparent pass-throughs.
def bypass_aug(layer):
    if 'random' in layer.name.lower() or 'flip' in layer.name.lower() or 'rotation' in layer.name.lower():
        print(f" -> Neutralizing: {layer.name}")
        # Replace with a linear activation (which does absolutely nothing to the data)
        return tf.keras.layers.Activation('linear', name=layer.name + "_bypassed")
    return layer

clean_model = tf.keras.models.clone_model(model, clone_function=bypass_aug)

# CRITICAL: We must copy your trained weights over to the cloned model!
clean_model.set_weights(model.get_weights())

print("3. Exporting clean model to standard format...")
clean_model.export('/content/saved_model_dir')

print("4. Converting to TensorFlow.js WITH QUANTIZATION (Float16)...")
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --quantize_float16 \
    /content/saved_model_dir \
    /content/tfjs_model

print("5. Verifying and Zipping...")
if os.path.exists('/content/tfjs_model/model.json'):
    !zip -r -q /content/tfjs_model.zip /content/tfjs_model
    print("SUCCESS! Conversion Complete. You can now download tfjs_model.zip.")
else:
    print("ERROR: The converter failed to generate the model.json.")

1. Loading original model...
2. Safely bypassing Data Augmentation layers...
3. Exporting clean model to standard format...
Saved artifact at '/content/saved_model_dir'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 299, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 15), dtype=tf.float32, name=None)
Captures:
  137874830711056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413518480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413510992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413509264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413505616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413513872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413513104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413511952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137874413512144: Ten

In [ ]:
!zip -r -q /content/tfjs_model.zip . -i /content/tfjs_model

from google.colab import files
files.download('/content/tfjs_model.zip')



zip error: Interrupted (aborting)


FileNotFoundError: Cannot find file: /content/tfjs_model.zip